# 🔬 IntelliCode-SL | Classifier SLM — Full Precision vs 8-bit

**What this notebook tests:**

| Config | Base Model | Adapter |
|--------|-----------|--------|
| **A** | Full precision (fp16, no quantization) | Full precision |
| **B** | 8-bit quantized | Full precision |

**Previous result (fp16 vs 4-bit):**
- Config A (fp16): 93.67% @ 158ms
- Config B (4-bit): 93.00% @ 219ms
- Verdict: fp16 won on both accuracy AND speed

**This test:** Does 8-bit close the gap vs full precision?  
**Test:** 50 tricky prompts × 6 categories = **300 total prompts**

> Run cells top to bottom. T4 GPU runtime required.

In [1]:
# ── Cell 1: Install ────────────────────────────────────────────
!pip install -q unsloth transformers peft accelerate bitsandbytes
print("✅ Dependencies installed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 117.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 106.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 117.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 81.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22

In [2]:
# ── Cell 2: Imports ────────────────────────────────────────────
import os, torch, time, gc, json
from collections import defaultdict
from google.colab import drive
from huggingface_hub import login
from unsloth import FastLanguageModel
from peft import PeftModel

print("✅ Imports done")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT FOUND")
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
✅ Imports done
GPU: Tesla T4
VRAM: 15.64 GB


In [ ]:
# ── Cell 3: Mount Drive + Login ────────────────────────────────
drive.mount("/content/drive")

HF_TOKEN     = "hf_XXXXXXXXXXXXXXXXXXXXXXXXXX"  # ← paste your token
ADAPTER_PATH = "/content/drive/MyDrive/IntelliCode-SL/adapters/classifier_adapter"
MODEL_NAME   = "Qwen/Qwen2.5-Coder-0.5B-Instruct"
MAX_SEQ_LEN  = 512

login(token=HF_TOKEN)
print("✅ Drive mounted + HF login done")

Mounted at /content/drive
✅ Drive mounted + HF login done


In [4]:
# ── Cell 4: 300 Tricky Test Prompts (50 per category) ──────────
# Identical test suite to the previous notebook for fair comparison

PROMPT_TEMPLATE = """### Instruction:
Classify the following user request into exactly one category:
debug, generate, modify, explain, document, unknown

### User Request:
{}

### Category:
"""

test_cases = {
    "debug": [
        ("Rewrite this buggy function so it actually works", "debug"),
        ("My function crashes with a KeyError, fix it and add error handling", "debug"),
        ("There's an off by one error in my loop, fix it and also add comments", "debug"),
        ("This code is broken, can you take a look and repair it", "debug"),
        ("The output is wrong on edge cases, please correct the logic", "debug"),
        ("I have a Python function that is supposed to calculate the average but it keeps throwing a ZeroDivisionError when the list is empty, can you fix the issue", "debug"),
        ("My recursive function works for small inputs but causes a RecursionError for large inputs, please fix this", "debug"),
        ("The sorting function I wrote produces correct output for most cases but fails when all elements are the same, fix this edge case", "debug"),
        ("I am getting an AttributeError on line 15 that says NoneType has no attribute strip, please resolve this", "debug"),
        ("My database query function throws an exception when the result set is empty, I need this fixed", "debug"),
        ("Fix the IndexError in my code and also make the variable names clearer", "debug"),
        ("There is a bug on line 8, correct it and then add a docstring", "debug"),
        ("The logic is wrong here, fix it and add a unit test too", "debug"),
        ("Debug this function and while you are at it improve the error messages", "debug"),
        ("My loop is infinite, fix the termination condition and optimize the loop body", "debug"),
        ("Getting TypeError: unsupported operand type in my calculator function", "debug"),
        ("ValueError is being raised when I call parse_date with certain inputs", "debug"),
        ("NameError: name result is not defined, what is wrong with my code", "debug"),
        ("My code throws RuntimeError: maximum recursion depth exceeded", "debug"),
        ("ImportError when trying to use my custom module, help me fix it", "debug"),
        ("Why does my function return the wrong value for negative inputs", "debug"),
        ("The test cases are failing for my binary search implementation", "debug"),
        ("My code does not behave as expected when the input is an empty string", "debug"),
        ("The program gives incorrect results when processing large numbers", "debug"),
        ("My merge function is dropping some elements, there must be a bug", "debug"),
        ("Fix the null pointer exception and also handle the case where the list is empty", "debug"),
        ("There are two bugs: the comparison operator is wrong and the loop starts at 1 instead of 0", "debug"),
        ("My function works but gives wrong results for floats, fix the precision issue", "debug"),
        ("The regex pattern is not matching correctly, fix it", "debug"),
        ("My API call fails with a 401 status, fix the authentication header", "debug"),
        ("Fix this broken code", "debug"),
        ("Something is wrong with this function", "debug"),
        ("This code has a bug", "debug"),
        ("My program crashes on startup", "debug"),
        ("The output is incorrect", "debug"),
        ("Resolve the segmentation fault in my C code", "debug"),
        ("My JavaScript function throws a ReferenceError", "debug"),
        ("The Java method throws NullPointerException unexpectedly", "debug"),
        ("SQL query returns no rows when it should return data", "debug"),
        ("My shell script exits with code 1 and I cannot find why", "debug"),
        ("The counter increments twice instead of once, fix this", "debug"),
        ("My list gets reversed when I do not want it to be", "debug"),
        ("The function skips the first element every time", "debug"),
        ("My dictionary is missing keys after the update operation", "debug"),
        ("The calculation is off by exactly one, there must be an error", "debug"),
        ("The function fails only on empty input, patch this", "debug"),
        ("Works fine in testing but crashes in production", "debug"),
        ("My code handles most cases but fails for very large numbers", "debug"),
        ("The function is correct but throws an exception when called twice", "debug"),
        ("Fix the race condition in my multithreaded code", "debug"),
    ],
    "generate": [
        ("Write a function to check if a string is a palindrome", "generate"),
        ("Create a class that implements a priority queue", "generate"),
        ("Write code to parse a JSON file and extract all email addresses", "generate"),
        ("Implement a simple REST API client using requests", "generate"),
        ("Write a function to convert a binary tree to a list", "generate"),
        ("I need a complete Python class for managing a library system that supports adding books, removing books, searching by title or author, checking out books, and returning books with proper error handling", "generate"),
        ("Write a decorator that limits how many times a function can be called per minute and raises an exception when the limit is exceeded", "generate"),
        ("Create a function that takes a list of transactions and returns a summary grouped by category with totals for each category", "generate"),
        ("Write a script that reads a CSV file, removes duplicate rows based on a given column, and writes the result to a new file", "generate"),
        ("Implement a thread-safe counter class that can be safely incremented and decremented from multiple threads simultaneously", "generate"),
        ("Build a function that takes existing code as a string and counts the number of functions defined in it", "generate"),
        ("Write a utility that wraps any function to add timing and logging automatically", "generate"),
        ("Create a class that behaves like a dictionary but also tracks how many times each key is accessed", "generate"),
        ("Write a function that takes a buggy string of code and returns a list of line numbers with syntax errors", "generate"),
        ("Implement a function that generates test cases for a given function signature", "generate"),
        ("Code a solution to find the longest common substring of two strings", "generate"),
        ("Write a function to detect if a directed graph has a cycle", "generate"),
        ("Implement Dijkstra's shortest path algorithm", "generate"),
        ("Write code to solve the coin change problem using dynamic programming", "generate"),
        ("Create a function to find the kth smallest element in an unsorted array", "generate"),
        ("Write a password generator", "generate"),
        ("Create a fibonacci sequence generator", "generate"),
        ("Write a function to flatten a nested dictionary", "generate"),
        ("Build a simple calculator class", "generate"),
        ("Write a URL validator function", "generate"),
        ("Write unit tests for a stack implementation", "generate"),
        ("Create a mock API server for testing", "generate"),
        ("Write a script to monitor CPU and memory usage", "generate"),
        ("Build a file diff utility that shows line by line differences", "generate"),
        ("Write a function to serialize and deserialize a binary tree", "generate"),
        ("From scratch, implement a hash map with chaining for collision resolution", "generate"),
        ("Build a complete tokenizer for simple arithmetic expressions", "generate"),
        ("Implement a bloom filter from scratch", "generate"),
        ("Create a simple state machine framework", "generate"),
        ("Write a connection pool implementation", "generate"),
        ("Write a function to paginate API results with cursor based pagination", "generate"),
        ("Create a simple in-memory job queue with worker threads", "generate"),
        ("Write a function to batch database inserts for performance", "generate"),
        ("Build a simple cache with TTL expiry", "generate"),
        ("Write a webhook signature verification function", "generate"),
        ("I want a new function that does X, Y and Z", "generate"),
        ("Can you code up a solution for this problem", "generate"),
        ("Put together a class that handles file uploads", "generate"),
        ("Come up with an implementation of the observer pattern", "generate"),
        ("Produce a working example of a producer consumer queue", "generate"),
        ("Write a function to normalize a list of numbers between 0 and 1", "generate"),
        ("Create a function to compute a rolling average over a sliding window", "generate"),
        ("Write code to group a list of objects by a given key", "generate"),
        ("Build a function that merges two sorted lists into one sorted list", "generate"),
        ("Write a function to detect outliers in a list using IQR", "generate"),
    ],
    "modify": [
        ("Add type hints to all functions in this file", "modify"),
        ("Refactor this code to use list comprehensions instead of for loops", "modify"),
        ("Add logging to every function in this module", "modify"),
        ("Optimize this function to reduce its time complexity", "modify"),
        ("Convert this synchronous code to use async await", "modify"),
        ("This code works but is very slow, make it faster", "modify"),
        ("The function is correct but lacks input validation, add it", "modify"),
        ("This code works but crashes on None input, handle that case", "modify"),
        ("My function is fine but it mutates the input list, fix the side effect", "modify"),
        ("The code runs but uses too much memory, optimize it", "modify"),
        ("Take this basic function and extend it to support batch processing", "modify"),
        ("Extend this class with three new utility methods", "modify"),
        ("Take this existing code and add pagination support to it", "modify"),
        ("Update this function to also support CSV input in addition to JSON", "modify"),
        ("Extend this API client to support authentication", "modify"),
        ("Refactor this large function into smaller single-responsibility functions", "modify"),
        ("Convert these repeated if-else chains into a strategy pattern", "modify"),
        ("Replace this mutable global state with dependency injection", "modify"),
        ("Refactor this class to follow the open closed principle", "modify"),
        ("Break this monolithic module into separate concerns", "modify"),
        ("The database queries in this function are N+1, fix the performance issue", "modify"),
        ("Add caching to this expensive computation", "modify"),
        ("Replace this O(n squared) algorithm with a more efficient approach", "modify"),
        ("Add connection pooling to this database module", "modify"),
        ("Reduce the memory footprint of this data processing function", "modify"),
        ("Harden this function against SQL injection", "modify"),
        ("Add input sanitization to this form handler", "modify"),
        ("Make this password comparison timing-safe", "modify"),
        ("Add rate limiting to this endpoint", "modify"),
        ("Encrypt the sensitive fields in this data model", "modify"),
        ("Add error handling to this function", "modify"),
        ("Make this code more readable", "modify"),
        ("Clean up this messy function", "modify"),
        ("Improve the performance of this code", "modify"),
        ("Update this code to use the new API", "modify"),
        ("Convert this class-based code to functional style", "modify"),
        ("Migrate this code from Python 2 syntax to Python 3", "modify"),
        ("Rewrite this using dataclasses instead of plain dictionaries", "modify"),
        ("Switch this from using raw SQL to using the ORM", "modify"),
        ("Convert this callback-based code to use promises", "modify"),
        ("Add retry logic with exponential backoff to this API call", "modify"),
        ("Add soft delete support to this repository class", "modify"),
        ("Add event hooks to this processor so callers can react to changes", "modify"),
        ("Add support for multiple output formats to this report generator", "modify"),
        ("Add circuit breaker pattern to this service client", "modify"),
        ("Make this function more idiomatic Python", "modify"),
        ("Simplify this complex nested condition", "modify"),
        ("Remove the code duplication in this module", "modify"),
        ("Make the variable names more descriptive", "modify"),
        ("Add assertions to validate the intermediate state in this algorithm", "modify"),
    ],
    "explain": [
        ("What does this recursive function do step by step", "explain"),
        ("Explain how this binary search implementation works", "explain"),
        ("Walk me through what this decorator is doing", "explain"),
        ("What is happening in this async await code", "explain"),
        ("Can you explain the logic behind this dynamic programming solution", "explain"),
        ("What is this function doing and why does it fail on empty input", "explain"),
        ("Explain what this code does and where the potential bug might be", "explain"),
        ("Walk me through this code and tell me if the logic looks correct", "explain"),
        ("What does this do and why might it be slow", "explain"),
        ("Explain this function to me, it seems to give wrong output sometimes", "explain"),
        ("What does each part of this code do, explain section by section", "explain"),
        ("Can you describe what this class and its methods are responsible for", "explain"),
        ("Break down this pipeline and explain each step", "explain"),
        ("Give me an overview of how this module works", "explain"),
        ("What is the purpose of each function in this file", "explain"),
        ("Explain why this sorting algorithm is O(n log n) and not O(n squared)", "explain"),
        ("What is the space complexity of this recursive approach", "explain"),
        ("Why does this code use a deque instead of a list here", "explain"),
        ("Explain why this memoization makes the function faster", "explain"),
        ("What is the purpose of the yield keyword in this generator", "explain"),
        ("What does this code do", "explain"),
        ("How does this work", "explain"),
        ("What is this function for", "explain"),
        ("Why is this written this way", "explain"),
        ("What does this return", "explain"),
        ("What design pattern is being implemented here", "explain"),
        ("Explain the observer pattern being used in this code", "explain"),
        ("What is the purpose of this context manager", "explain"),
        ("Explain what this lambda function is doing", "explain"),
        ("What is the role of this middleware function", "explain"),
        ("I do not understand this code, can you explain it simply", "explain"),
        ("This code is confusing to me, break it down", "explain"),
        ("Can you help me understand what this algorithm is doing", "explain"),
        ("Explain this to me like I am a beginner", "explain"),
        ("What is the intuition behind this approach", "explain"),
        ("What is the purpose of the try except block in this code", "explain"),
        ("Why does this function call super() here", "explain"),
        ("What does the double underscore in this method name mean", "explain"),
        ("Why is this property decorated with @staticmethod", "explain"),
        ("What does the walrus operator do in this if statement", "explain"),
        ("What will happen when this function is called with an empty list", "explain"),
        ("Trace through this code with input x equals 5", "explain"),
        ("What is the output of this code", "explain"),
        ("What side effects does this function have", "explain"),
        ("What happens if I call this method twice in a row", "explain"),
        ("Why is a set used here instead of a list", "explain"),
        ("Why does this algorithm start from the end of the array", "explain"),
        ("Why is this variable initialized to None here", "explain"),
        ("Why does this recursive call pass n minus 1", "explain"),
        ("Why is this lock needed here", "explain"),
    ],
    "document": [
        ("Write docstrings for all functions in this file", "document"),
        ("Generate a README for this project", "document"),
        ("Add inline comments explaining the logic of this algorithm", "document"),
        ("Create API documentation for these endpoints", "document"),
        ("Write a docstring with Args Returns and Raises for this function", "document"),
        ("Add comments explaining what each section of this code does", "document"),
        ("Document this class so other developers understand how to use it", "document"),
        ("Add documentation that explains the purpose of each method", "document"),
        ("Write comments that describe the algorithm step by step", "document"),
        ("Create documentation that makes this code self-explanatory", "document"),
        ("Generate Google style docstrings for this module", "document"),
        ("Write NumPy style documentation for this class", "document"),
        ("Create JSDoc comments for this JavaScript file", "document"),
        ("Write Sphinx-compatible documentation for this package", "document"),
        ("Generate OpenAPI specification for this REST API", "document"),
        ("Write a README with installation usage and examples sections", "document"),
        ("Create a README that explains what this library does and how to contribute", "document"),
        ("Generate a README with badges for the CI status and coverage", "document"),
        ("Write a getting started guide for this SDK", "document"),
        ("Create a quickstart tutorial for this framework", "document"),
        ("Add inline comments to this complex sorting algorithm", "document"),
        ("Comment this regex pattern so it is clear what each part matches", "document"),
        ("Add comments to explain the purpose of each variable in this function", "document"),
        ("Insert comments at every non-obvious step of this algorithm", "document"),
        ("Add section headers and comments throughout this long function", "document"),
        ("Document this function", "document"),
        ("Add docstrings to this class", "document"),
        ("Write documentation for this code", "document"),
        ("Comment this code", "document"),
        ("Create docs for this module", "document"),
        ("Write release notes for version 2.0", "document"),
        ("Generate a changelog entry for the new features added this sprint", "document"),
        ("Document the breaking changes in this API update", "document"),
        ("Write a migration guide from version 1 to version 2", "document"),
        ("Create a deprecation notice for this old function", "document"),
        ("Write user documentation for this CLI tool", "document"),
        ("Create a tutorial walkthrough for this feature", "document"),
        ("Write a how-to guide for integrating this library", "document"),
        ("Generate example usage snippets for this module", "document"),
        ("Write a troubleshooting guide for common errors", "document"),
        ("Document the architecture and design decisions in this module", "document"),
        ("Write an architecture decision record for using this approach", "document"),
        ("Document the data flow through this pipeline", "document"),
        ("Create a developer onboarding guide for this codebase", "document"),
        ("Write technical documentation describing the system design", "document"),
        ("Make this code more self-documenting", "document"),
        ("Add docstrings so this passes pydocstyle checks", "document"),
        ("The code has no comments at all, add appropriate documentation", "document"),
        ("Document all public methods in this class", "document"),
        ("Add type documentation for all parameters in this module", "document"),
    ],
    "unknown": [
        ("What is the difference between TCP and UDP", "unknown"),
        ("Explain the CAP theorem in distributed systems", "unknown"),
        ("What is the difference between a process and a thread", "unknown"),
        ("How does garbage collection work in Python", "unknown"),
        ("What is the difference between stack and heap memory", "unknown"),
        ("Compare Python vs JavaScript for backend development", "unknown"),
        ("What are the pros and cons of REST vs GraphQL", "unknown"),
        ("Should I use PostgreSQL or MongoDB for this use case", "unknown"),
        ("What is the difference between Docker and a virtual machine", "unknown"),
        ("Compare microservices versus monolithic architecture", "unknown"),
        ("What programming language should I learn first", "unknown"),
        ("How do I prepare for a software engineering interview", "unknown"),
        ("What skills do I need to become a backend developer", "unknown"),
        ("How long does it take to learn Python", "unknown"),
        ("What certifications are useful for cloud engineers", "unknown"),
        ("What is event driven architecture", "unknown"),
        ("Explain the SOLID principles", "unknown"),
        ("What is domain driven design", "unknown"),
        ("What is clean architecture", "unknown"),
        ("Explain the twelve factor app methodology", "unknown"),
        ("What is Kubernetes used for", "unknown"),
        ("How does Redis work", "unknown"),
        ("What is Kafka and when should I use it", "unknown"),
        ("What is the purpose of a load balancer", "unknown"),
        ("What is a CDN and how does it work", "unknown"),
        ("What is OAuth2 and how does it work", "unknown"),
        ("Explain public key cryptography", "unknown"),
        ("What is a SQL injection attack", "unknown"),
        ("How does JWT authentication work", "unknown"),
        ("What is zero trust security", "unknown"),
        ("What is the time complexity of quicksort in the worst case", "unknown"),
        ("Explain Big O notation", "unknown"),
        ("What is the difference between BFS and DFS", "unknown"),
        ("When should I use dynamic programming", "unknown"),
        ("What is the P vs NP problem", "unknown"),
        ("What is overfitting in machine learning", "unknown"),
        ("Explain the difference between supervised and unsupervised learning", "unknown"),
        ("What is a transformer model", "unknown"),
        ("What is gradient descent", "unknown"),
        ("Explain transfer learning", "unknown"),
        ("What is recursion and why is it useful", "unknown"),
        ("What are design patterns and why do they matter", "unknown"),
        ("What is the difference between concurrency and parallelism", "unknown"),
        ("What is a closure in programming", "unknown"),
        ("What is dependency injection", "unknown"),
        ("What are the best practices for writing clean code", "unknown"),
        ("How should I structure a large Python project", "unknown"),
        ("What is the best way to handle errors in an API", "unknown"),
        ("When should I use a class versus a function", "unknown"),
        ("What are the most common code smells to avoid", "unknown"),
    ]
}

print(f"✅ Test suite ready — {sum(len(v) for v in test_cases.values())} total prompts")
for cat, cases in test_cases.items():
    print(f"   {cat:<10} : {len(cases)} prompts")

✅ Test suite ready — 300 total prompts
   debug      : 50 prompts
   generate   : 50 prompts
   modify     : 50 prompts
   explain    : 50 prompts
   document   : 50 prompts
   unknown    : 50 prompts


In [5]:
# ── Cell 5: Inference + Evaluation Helpers ─────────────────────
VALID_LABELS = {"debug", "generate", "modify", "explain", "document", "unknown"}
CATEGORIES   = ["debug", "generate", "modify", "explain", "document", "unknown"]

def predict(model, tokenizer, prompt):
    test_input = PROMPT_TEMPLATE.format(prompt)
    inputs = tokenizer(test_input, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=10, do_sample=False,
            temperature=None, top_p=None
        )
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    raw = result.split("### Category:")[-1].strip().split()[0].lower()
    for v in VALID_LABELS:
        if raw.startswith(v):
            return v
    return raw

def evaluate(model, tokenizer, test_cases_dict, config_name):
    print(f"\n{'='*55}\n  Evaluating: {config_name}\n{'='*55}")
    results = []
    per_cat  = defaultdict(lambda: {"correct": 0, "total": 0, "wrong": []})
    confusion = defaultdict(lambda: defaultdict(int))
    total_time = 0
    all_cases = [(p, l) for cases in test_cases_dict.values() for p, l in cases]

    for i, (prompt, expected) in enumerate(all_cases):
        t0 = time.time()
        predicted = predict(model, tokenizer, prompt)
        elapsed = time.time() - t0
        total_time += elapsed
        correct = predicted == expected
        per_cat[expected]["total"] += 1
        if correct:
            per_cat[expected]["correct"] += 1
        else:
            per_cat[expected]["wrong"].append(
                {"prompt": prompt[:60], "expected": expected, "predicted": predicted}
            )
        confusion[expected][predicted] += 1
        results.append({"prompt": prompt, "expected": expected, "predicted": predicted, "correct": correct})
        if (i + 1) % 50 == 0:
            done = sum(1 for r in results if r["correct"])
            print(f"  [{i+1:>3}/300] Running accuracy: {done/(i+1)*100:.1f}%")

    total_correct = sum(1 for r in results if r["correct"])
    accuracy = total_correct / len(results) * 100
    avg_time = total_time / len(results)

    print(f"\n  ✅ Overall Accuracy : {accuracy:.2f}% ({total_correct}/{len(results)})")
    print(f"  ⏱  Avg inference    : {avg_time*1000:.1f} ms/prompt")
    print(f"\n  {'Category':<12} {'Correct':>8} {'Total':>7} {'Accuracy':>10}")
    print(f"  {'-'*40}")
    for cat in CATEGORIES:
        s = per_cat[cat]
        acc = s["correct"] / s["total"] * 100 if s["total"] else 0
        print(f"  {cat:<12} {s['correct']:>8} {s['total']:>7} {acc:>9.1f}%")

    return {
        "config": config_name, "accuracy": accuracy,
        "total_correct": total_correct, "total": len(results),
        "avg_time_ms": avg_time * 1000,
        "per_category": {k: dict(v) for k, v in per_cat.items()},
        "confusion": {k: dict(v) for k, v in confusion.items()},
        "results": results
    }

print("✅ Helpers ready")


✅ Helpers ready


In [6]:
# ── Cell 6: Config A — Full Precision (fp16) ───────────────────
import os
os.environ['UNSLOTH_USE_MODELSCOPE'] = '1'

print("Loading Config A: Full precision (fp16) base + full adapter...")

model_A, tokenizer_A = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = torch.float16,
    load_in_4bit   = False,
    token          = HF_TOKEN,
)
model_A = PeftModel.from_pretrained(model_A, ADAPTER_PATH, torch_dtype=torch.float16)
model_A = model_A.to("cuda")
FastLanguageModel.for_inference(model_A)

print("✅ Config A loaded")
print(f"   VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

Loading Config A: Full precision (fp16) base + full adapter...
==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/266 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/764 [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

✅ Config A loaded
   VRAM used: 1.05 GB


In [ ]:
# ── Cell 7: Evaluate Config A ──────────────────────────────────
results_A = evaluate(model_A, tokenizer_A, test_cases, "Config A: fp16 Full Precision")


In [ ]:
# ── Cell 8: Save results_A + Unload ────────────────────────────
import json, os

# Serialize for JSON (defaultdict -> dict)
save_A = {
    "config": results_A["config"],
    "accuracy": results_A["accuracy"],
    "total_correct": results_A["total_correct"],
    "total": results_A["total"],
    "avg_time_ms": results_A["avg_time_ms"],
    "per_category": {k: dict(v) for k, v in results_A["per_category"].items()},
    "confusion": {k: dict(v) for k, v in results_A["confusion"].items()},
    "results": results_A["results"]
}

save_path = "/content/drive/MyDrive/IntelliCode-SL/benchmarks/results_A_fp16.json"
os.makedirs(os.path.dirname(save_path), exist_ok=True)
with open(save_path, "w") as f:
    json.dump(save_A, f, indent=2)
print(f"✅ results_A saved → {save_path}")

del model_A, tokenizer_A
gc.collect()
torch.cuda.empty_cache()
print(f"✅ Config A unloaded | VRAM now: {torch.cuda.memory_allocated()/1e9:.2f} GB")
